# 📊 Monitoring & Observability — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Real-world scenarios, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---

**Topics**
1. The Three Pillars of Observability — Logs, Metrics, Traces
2. The Monitoring Mental Model — SLOs, SLIs, SLAs
3. Prometheus — Pull-Based Metrics
4. Grafana — Visualization and Alerting
5. The ELK Stack — Log Aggregation
6. Distributed Tracing — Jaeger and OpenTelemetry
7. The On-Call Engineer's Toolkit
8. Real-World: How Netflix and Google Observe Their Systems
9. The Observability Architect's Design Framework

---
## 1 · The Three Pillars of Observability

### 🧠 Mental Model — *Instrumentation as a First-Class Concern*

> **Monitoring tells you THAT something is broken. Observability tells you WHY it's broken — from the outside, by examining the system's outputs, without adding new code. The three pillars are your eyes and ears: metrics show trends, logs show events, traces show the journey of a request. A system missing any pillar has a blind spot.**

**WHY observability matters:** The difference between a 5-minute incident and a 5-hour incident is almost always the quality of your observability. Engineers debugging without metrics/logs/traces are guessing. Engineers with full observability are following a trail.

### The Three Pillars

```
METRICS (time-series numbers)
  "What is happening RIGHT NOW at system level?"
  
  Examples:
    http_requests_total{method="POST", status="500"} 42
    http_request_duration_seconds{p99} 0.847
    node_memory_used_bytes 4294967296
    active_connections 1024
  
  Tools: Prometheus, Datadog, CloudWatch, VictoriaMetrics
  When to use: Alerting, dashboards, capacity planning, SLO tracking
  Limitation: High cardinality (user_id in labels) kills Prometheus

LOGS (timestamped event records)
  "What happened and what was the context?"
  
  Examples:
    {"timestamp": "2024-01-15T10:23:45Z", "level": "ERROR",
     "service": "payment", "user_id": "usr_123",
     "message": "Stripe charge failed", "amount": 9900,
     "error": "card_declined", "trace_id": "abc123"}
  
  Tools: ELK Stack (Elasticsearch, Logstash, Kibana), Loki, Splunk
  When to use: Debugging specific incidents, audit trails, compliance
  Limitation: High volume = high cost; unstructured logs are hard to query

TRACES (request journey across services)
  "How did this specific request travel through the system?"
  
  Example: A single payment request:
    ├── API Gateway (2ms)
    │   └── auth-service (5ms)
    ├── payment-service (847ms total) ← SLOW
    │   ├── DB query (2ms)
    │   ├── Stripe API call (820ms) ← ROOT CAUSE
    │   └── notification queue (8ms)
    └── response (854ms total)
  
  Tools: Jaeger, Zipkin, DataDog APM, AWS X-Ray
  When to use: Diagnosing latency in distributed systems
  Limitation: Requires instrumentation across all services; sampling needed at scale
```

### The Correlation Model — The Key to Fast Debugging

```
The three pillars become 10× more powerful when they are CORRELATED:

1. Metric alert fires: p99 latency > 500ms on payment-service
2. Dashboard shows: spike at 14:23 UTC
3. Jump to logs: filter logs for payment-service at 14:23
4. Log shows: Stripe timeout errors, trace_id: abc123def456
5. Jump to trace abc123def456: Stripe API call took 820ms
6. Root cause: Stripe API latency spike. Fix: add circuit breaker.

Time from alert to root cause: ~5 minutes (with correlation)
Time from alert to root cause: ~2 hours (without correlation, grepping logs)

Key: every log entry must include trace_id so you can jump from log → trace.
     Every trace must include service name so you can jump from trace → metrics.
```

### 🌍 Real-World: Google's Monarch + Dapper
Google operates two massive observability systems:
- **Monarch:** Time-series metrics system processing 1 trillion data points per minute
- **Dapper:** Distributed tracing, inspired Jaeger, Zipkin, and the OpenTracing standard

The core insight from Google's SRE book: **"The most important feature of a monitoring system is that it answers: Are things working? If not, what's wrong? If I need to fix it, how do I do so quickly?"**

---
## 2 · SLOs, SLIs, SLAs — The Language of Reliability

### 🧠 Mental Model — *Reliability as a Feature with a Budget*

> **SLOs (Service Level Objectives) are the targets. SLIs (Service Level Indicators) are the measurements. SLAs (Service Level Agreements) are the contracts. The Error Budget is the inverse of your SLO — it's how much you're ALLOWED to fail. When your error budget runs out, you stop shipping features and fix reliability. This is the Google SRE model.**

### The Hierarchy

```
SLA (contractual commitment to customers):
  "We guarantee 99.9% availability. If we breach this, you get credits."
  This is the external-facing promise.

SLO (internal target, stricter than SLA):
  "Our INTERNAL target is 99.95% availability."
  The buffer between SLO and SLA gives you time to react before breaching the SLA.

SLI (the actual measurement):
  "Current availability (measured): 99.97%"
  The metric you use to evaluate your SLO.

Error Budget:
  = 1 - SLO = 100% - 99.95% = 0.05% downtime allowed per month
  0.05% of 30 days = 21.6 minutes of allowed downtime per month

Error Budget Burn Rate:
  How fast you're consuming the budget.
  Burn rate 1x: consuming budget at SLO rate (sustainable)
  Burn rate 14.4x: will exhaust monthly budget in 2 hours → PAGE IMMEDIATELY
  Burn rate 6x:   will exhaust monthly budget in 5 days → PAGE SOON
```

### The Four Golden Signals (Google SRE)

| Signal | What it measures | Example SLI | Example SLO |
|---|---|---|---|
| **Latency** | How long requests take | p99 request duration | p99 < 500ms |
| **Traffic** | How much demand | requests per second | — (for sizing, not SLO) |
| **Errors** | How often requests fail | % of 5xx responses | < 0.1% error rate |
| **Saturation** | How full the system is | CPU utilization, queue depth | CPU < 80% |

```
Start here: the Four Golden Signals tell you 95% of what you need to know.
Add custom SLIs for your specific SLAs once the basics are covered.
```

In [ ]:
"""
SLO Error Budget Tracker
========================
Implements the Google SRE error budget model:
- Tracks availability over a rolling window
- Calculates error budget remaining
- Computes burn rate
- Generates multi-window alerts (like Google's alerting strategy)

This is the kind of tool SRE teams build to operationalize SLOs.
"""
from __future__ import annotations
import math
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import List, Tuple
import random

In [ ]:
@dataclass
class RequestWindow:
    timestamp: datetime
    total_requests: int
    good_requests: int

    @property
    def bad_requests(self) -> int:
        return self.total_requests - self.good_requests

    @property
    def success_rate(self) -> float:
        if self.total_requests == 0:
            return 1.0
        return self.good_requests / self.total_requests


class SLOTracker:
    """
    Tracks SLO compliance and error budget consumption.
    Implements the Google multi-window alerting strategy.
    """

    def __init__(self, slo: float, window_days: int = 30):
        self.slo = slo
        self.window_days = window_days
        self.windows: List[RequestWindow] = []

    def record(self, total: int, good: int, timestamp: datetime = None) -> None:
        ts = timestamp or datetime.now()
        self.windows.append(RequestWindow(ts, total, good))

    def _windows_in_period(self, hours: int) -> List[RequestWindow]:
        cutoff = datetime.now() - timedelta(hours=hours)
        return [w for w in self.windows if w.timestamp >= cutoff]

    def _sli(self, windows: List[RequestWindow]) -> float:
        total = sum(w.total_requests for w in windows)
        good  = sum(w.good_requests for w in windows)
        return good / total if total > 0 else 1.0

    def error_budget(self) -> dict:
        all_windows = self._windows_in_period(self.window_days * 24)

        total_requests = sum(w.total_requests for w in all_windows)
        good_requests  = sum(w.good_requests for w in all_windows)
        bad_requests   = total_requests - good_requests

        allowed_bad    = total_requests * (1 - self.slo)
        consumed_pct   = (bad_requests / allowed_bad * 100) if allowed_bad > 0 else 0
        remaining_pct  = max(0, 100 - consumed_pct)

        current_sli    = self._sli(all_windows)

        return {
            "slo": self.slo,
            "current_sli": current_sli,
            "total_requests": total_requests,
            "bad_requests": bad_requests,
            "allowed_bad_requests": allowed_bad,
            "budget_consumed_pct": consumed_pct,
            "budget_remaining_pct": remaining_pct,
            "is_slo_breached": current_sli < self.slo,
        }

    def burn_rate(self, short_hours: float = 1.0, long_hours: float = 6.0) -> dict:
        """
        Multi-window burn rate (Google's recommended alerting strategy).
        Alert when BOTH short and long windows show high burn rate.
        This reduces false positives while catching real incidents quickly.
        """
        short_sli = self._sli(self._windows_in_period(int(short_hours * 60) // 60 + 1))
        long_sli  = self._sli(self._windows_in_period(int(long_hours)))

        error_rate   = lambda sli: 1 - sli
        budget_error = 1 - self.slo

        short_burn = error_rate(short_sli) / budget_error if budget_error > 0 else 0
        long_burn  = error_rate(long_sli)  / budget_error if budget_error > 0 else 0

        # Alert thresholds (from Google SRE book)
        critical = short_burn > 14.4 and long_burn > 14.4  # 2-hr budget exhaustion
        warning  = short_burn > 6 and long_burn > 6         # 5-day budget exhaustion

        return {
            "short_window_burn_rate": short_burn,
            "long_window_burn_rate": long_burn,
            "alert_critical": critical,
            "alert_warning": warning,
        }


# Simulate 30 days of traffic with an incident on day 20
random.seed(42)
tracker = SLOTracker(slo=0.999, window_days=30)  # 99.9% SLO
base_ts = datetime.now() - timedelta(days=30)

for day in range(30):
    ts = base_ts + timedelta(days=day)
    if day == 20:  # simulate an incident on day 20
        total = 100_000
        good  = 94_000  # 6% error rate — terrible!
    else:
        total = 100_000
        good  = int(100_000 * (0.9997 + random.uniform(-0.0001, 0.0001)))
    tracker.record(total, good, timestamp=ts)

budget = tracker.error_budget()
burn   = tracker.burn_rate()

print("=== SLO Error Budget Report (30-day window) ===")
print(f"  SLO Target:          {budget['slo']:.1%}")
print(f"  Current SLI:         {budget['current_sli']:.4%}")
print(f"  Total requests:      {budget['total_requests']:,}")
print(f"  Bad requests:        {budget['bad_requests']:,}")
print(f"  Allowed bad:         {budget['allowed_bad_requests']:,.0f}")
print(f"  Budget consumed:     {budget['budget_consumed_pct']:.1f}%")
print(f"  Budget remaining:    {budget['budget_remaining_pct']:.1f}%")
print(f"  SLO breached:        {'⚠️  YES' if budget['is_slo_breached'] else '✅ No'}")
print(f"\n  Burn Rate Analysis:")
print(f"  Short-window burn:   {burn['short_window_burn_rate']:.2f}×")
print(f"  Long-window burn:    {burn['long_window_burn_rate']:.2f}×")
print(f"  🔴 CRITICAL alert:    {burn['alert_critical']}")
print(f"  🟡 WARNING alert:     {burn['alert_warning']}")

---
## 3 · Prometheus — Pull-Based Metrics Architecture

### 🧠 Mental Model — *Prometheus Scrapes; It Is Not Pushed To*

> **Prometheus is a pull-based metrics system. Instead of your application pushing metrics to a central server, Prometheus periodically SCRAPES an HTTP endpoint (`/metrics`) on each target. The advantage: if a target goes down, Prometheus immediately knows (the scrape fails). With push-based systems, a dead target just... stops pushing, and you have no baseline to detect the silence.**

### Architecture

```
Prometheus Server
  ├── Service Discovery (Kubernetes, Consul, EC2 tags)
  │   "Find all targets to scrape"
  │
  ├── Scrape Loop (every 15s by default)
  │   GET http://payment-service:8000/metrics
  │   GET http://node-exporter:9100/metrics
  │   GET http://postgres-exporter:9187/metrics
  │
  ├── TSDB (time-series database)
  │   Stores metrics with labels and timestamps
  │   Retention: 15 days default (send to Thanos/Cortex for long-term)
  │
  ├── PromQL (query language)
  │   rate(http_requests_total[5m])  ← requests per second
  │   histogram_quantile(0.99, http_request_duration_seconds_bucket)
  │
  └── Alertmanager
      Routes alerts → Slack, PagerDuty, email based on rules
```

### Metric Types

| Type | What it is | Example |
|---|---|---|
| **Counter** | Monotonically increasing (never decreases) | `http_requests_total` |
| **Gauge** | Value that goes up and down | `active_connections`, `temperature` |
| **Histogram** | Counts observations in configurable buckets | `http_request_duration_seconds` |
| **Summary** | Streaming quantiles (less flexible) | Use histogram instead |

```
The CRUCIAL rule for counters:
  Never query a counter directly — it shows monotonically increasing values.
  Always use rate() or increase() to get the per-second rate:

  ❌ WRONG:  http_requests_total{status="500"}  → 1,234,567 (meaningless)
  ✅ RIGHT:  rate(http_requests_total{status="500"}[5m])  → 0.34/s (meaningful)

  increase(http_requests_total{status="500"}[1h])  → errors in the last hour
```

### Production Prometheus Setup — Kubernetes

```yaml
# Using kube-prometheus-stack (Helm chart)
# This installs: Prometheus + Alertmanager + Grafana + node-exporter
# + kube-state-metrics + all default Kubernetes dashboards and alerts

helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
helm install prometheus prometheus-community/kube-prometheus-stack \
  --namespace monitoring \
  --create-namespace \
  --values prometheus-values.yaml

# Add /metrics to your FastAPI app:
from prometheus_fastapi_instrumentator import Instrumentator
Instrumentator().instrument(app).expose(app)  # adds GET /metrics

# Tell Prometheus to scrape your service (via ServiceMonitor CRD):
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: payment-service
  labels:
    release: prometheus  # must match kube-prometheus-stack selector
spec:
  selector:
    matchLabels:
      app: payment-service
  endpoints:
    - port: http
      path: /metrics
      interval: 15s
```

### Essential PromQL Queries

```promql
# Error rate (% of 5xx responses)
sum(rate(http_requests_total{status=~"5.."}[5m])) by (service)
/
sum(rate(http_requests_total[5m])) by (service)
* 100

# p99 latency
histogram_quantile(0.99, 
  sum(rate(http_request_duration_seconds_bucket[5m])) by (le, service)
)

# CPU usage per pod
sum(rate(container_cpu_usage_seconds_total{container!=""}[5m])) by (pod)

# Memory OOM events
kube_pod_container_status_last_terminated_reason{reason="OOMKilled"}

# Node disk pressure
(node_filesystem_size_bytes - node_filesystem_avail_bytes) 
/ node_filesystem_size_bytes * 100 > 80
```

---

## 4 · The ELK Stack — Log Aggregation at Scale

### 🧠 Mental Model — *A Google Search Engine for Your Logs*

> **Elasticsearch is a distributed search engine. Logstash (or Fluent Bit) ships logs to it. Kibana is the UI. Together, they turn your distributed system's log output into a searchable, filterable, analysable corpus — like Google for your application's behavior. The key insight: structured JSON logs are 10× more useful than unstructured text logs.**

### Architecture

```
Application Containers
  │  (log to stdout/stderr — Docker/K8s captures these)
  │
  ▼
Fluent Bit (DaemonSet on each K8s node — lightweight log shipper)
  - Collects logs from all pods on the node
  - Parses and enriches: adds pod name, namespace, node, timestamp
  - Ships to Elasticsearch (or Logstash for heavy processing)
  │
  ▼
Elasticsearch Cluster (3+ nodes for HA)
  - Indexes and stores logs
  - Full-text search, aggregations, filtering
  - Retention: 30 days hot, 90 days warm (ILM policies)
  │
  ▼
Kibana
  - Search: filter by service, level, time range
  - Dashboards: error rate over time, top error messages
  - Alerts: trigger when error count > threshold
```

### Structured Logging — The Foundation

```python
# ❌ BEFORE: Unstructured logging (impossible to query at scale)
import logging
logging.error(f"Payment failed for user {user_id}: {str(error)}")
# Log: 'Payment failed for user usr_123: Card declined'
# To find all failures: grep 'Payment failed' logs/* (slow, brittle)

# ✅ AFTER: Structured JSON logging (queryable in ELK/Loki)
import structlog

log = structlog.get_logger()
log.error(
    "payment.failed",          # structured event name — queryable!
    user_id=user_id,
    amount=amount,
    currency=currency,
    error_code=error.code,
    error_message=str(error),
    stripe_request_id=response.id,
    trace_id=trace.get_current_span().get_span_context().trace_id,
    duration_ms=duration_ms,
)
# Log: {"event": "payment.failed", "user_id": "usr_123", "amount": 9900,
#        "error_code": "card_declined", "trace_id": "abc123..."}
# In Kibana: filter error_code='card_declined' AND amount>10000
# → Instantly find all high-value declined transactions
```

---

## 5 · The Observability Architect's Design Framework

### The Observability Checklist for Any New Service

```
METRICS:
  □ /metrics endpoint exposed (Prometheus scrape)
  □ Four Golden Signals instrumented
  □ Business metrics (orders processed, revenue, users signed up)
  □ SLO dashboards created
  □ Alerts on SLO burn rate (not just raw thresholds)

LOGS:
  □ Structured JSON logging (not unstructured text)
  □ Log levels used correctly (DEBUG local only, INFO prod, ERROR for real errors)
  □ trace_id in every log entry
  □ No PII in logs (GDPR/compliance)
  □ Log retention policy set

TRACES:
  □ OpenTelemetry SDK integrated
  □ Context propagation to all downstream calls
  □ Sampling rate appropriate (100% in dev, 10% in prod, adaptive)

ALERTING:
  □ Alerts route to correct team (not broadcast to everyone)
  □ Alert has runbook link (not just a vague 'something is wrong')
  □ Alerts are actionable (if engineer can't do anything, alert shouldn't fire)
  □ Noise budget: < 2 actionable pages per on-call engineer per week

DASHBOARDS:
  □ Service overview dashboard (RED: Rate, Errors, Duration)
  □ Infrastructure dashboard (CPU, Memory, Disk, Network)
  □ Business dashboard (product-level KPIs)
  □ Dashboards linked from runbooks
```

### The On-Call Engineer's 5-Minute Framework

```
When an alert fires:

1. Is it real? (Check: is the alert flapping? Is it a known issue?)
2. What's the impact? (# users affected, which features broken)
3. Metrics: Which services are showing degradation? Since when?
4. Logs: What errors appear at that timestamp? What changed?
5. Traces: Find a slow trace — what's the bottleneck?
6. Correlate: Did a deployment happen before the incident?
7. Mitigate first: rollback? circuit breaker? feature flag off?
8. Root cause later (during business hours, not at 3 AM)
9. Post-mortem: blameless, focus on systemic improvements
```